In [0]:

%load_ext autoreload
%autoreload 2 

from pyspark.sql import SparkSession
from datetime import datetime, timezone
import uuid 
from src.nyc_taxi_silver import NYC_Taxi_Silver_Loader 
import logging 

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

# 假设这是你的 Databricks Notebook 或 PySpark 提交脚本的入口
spark = SparkSession.builder.appName("NYCTaxi_Silver_Processing").getOrCreate()

# 1. 定义你的环境参数
BRONZE_TABLE = "nyc.process_bronze.brz_yellow_nyc_taxi"

TARGET_YYYYMM = "201001" # 目标处理月份
# 实际生产中通常由调度工具(如 Airflow/Databricks Workflow)传入
RUN_ID = f"nyc_yellow_silver_run_{TARGET_YYYYMM}_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}_{uuid.uuid4().hex[:8]}"

TARGET_TABLE = "nyc.process_silver.silver_yellow_taxi"

# 2. 实例化我们刚才优化的 Loader 类
silver_loader = NYC_Taxi_Silver_Loader(
    spark=spark, 
    run_id=RUN_ID, 
    target_table=TARGET_TABLE
)

# 3. 读取 Bronze 层数据 (按需过滤)
# 最佳实践：不要全表读取（spark.read.table），一定要带上下推过滤(Predicate Pushdown)
bronze_df = spark.table(BRONZE_TABLE).filter(f"YYYYMM = {TARGET_YYYYMM}")


# 4. 执行处理流程
silver_loader.process(bronze_df)

print(f"Pipeline finished successfully for run_id: {RUN_ID}")



In [0]:
%sql
select * from nyc.process_silver.pipeline_metrics 

In [0]:
%sql
DROP TABLE IF EXISTS nyc.process_silver.silver_yellow_taxi;
DROP TABLE IF EXISTS nyc.process_silver.silver_yellow_taxi_quarantine;
DROP TABLE audit.pipeline_metrics;

In [0]:
# 我们只挑选关键的列来展示，方便肉眼查看
display(
    dq_result_df.select(
        "is_valid", 
        "violated_rules", 
        "passenger_count", 
        "total_amount", 
        "duration_min"
    )
)

In [0]:
%pip install -q pytest

In [0]:
%pip install chispa

In [0]:
import sys
import os
import pytest

# 2. 封印字节码缓存 (解决之前的 Errno 95 权限报错)
sys.dont_write_bytecode = True

# 3. 智能路径导航：确保我们站在项目根目录下执行
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    os.chdir("..")
    print(f"✅ 已切换到根目录: {os.getcwd()}")

# 4. 把根目录注入到 Python 视野中 (替代 PYTHONPATH=.)
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 5. 在拥有完整高级权限的 Python 进程中，直接引爆 Pytest！
print("🚀 启动工业级单元测试流水线...\n" + "="*40)
#retcode = pytest.main(["tests/", "-v", "-p", 
# "no:cacheprovider"])
retcode = pytest.main([
    "tests/", 
    "-vv",               # 极度详细模式
    "--tb=long",         # 显示完整的错误堆栈，不要缩略
    "-s",                # 允许打印 print 语句
    "-p", "no:cacheprovider"
])


In [0]:
# clean memory
dbutils.library.restartPython()

In [0]:
import sys
import os
import pytest

# 2. 封印字节码缓存 (解决之前的 Errno 95 权限报错)
sys.dont_write_bytecode = True

# 3. 智能路径导航：确保我们站在项目根目录下执行
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    os.chdir("..")
    print(f"✅ 已切换到根目录: {os.getcwd()}")

# 4. 把根目录注入到 Python 视野中 (替代 PYTHONPATH=.)
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 5. 在拥有完整高级权限的 Python 进程中，直接引爆 Pytest！
print("🚀 启动工业级单元测试流水线 (单测精准狙击模式)...\n" + "="*40)

retcode = pytest.main([
    # 核心修改：用具体的路径和节点，替换掉原来的 "tests/"
    "tests/test_silver_integration.py::TestSilverIntegration::test_full_pipeline_and_partition_overwrite", 
    "-vv",               # 极度详细模式
    "--tb=long",         # 显示完整的错误堆栈，不要缩略
    "-s",                # 允许打印 print 语句
    "-p", "no:cacheprovider"
])